<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_04_2_schedule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>


# T81-558: Anwendungen tiefer neuronaler Netzwerke

**Modul 4: Training für tabellarische Daten**

- Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
– Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).


# Modul 4 Material

- Teil 4.1: Verwenden der K-fachen Kreuzvalidierung mit PyTorch [[Video]](https://www.youtube.com/watch?v=Q8ZQNvZwsNE&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=Q8ZQNvZwsNE&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- **Teil 4.2: Trainingspläne für PyTorch** [[Video]](https://www.youtube.com/watch?v=lMMlbmfvKDQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=lMMlbmfvKDQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.3: Dropout-Regularisierung [[Video]](https://www.youtube.com/watch?v=4ixjgw6Q42U&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=4ixjgw6Q42U&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.4: Batch-Normalisierung [[Video]](https://www.youtube.com/watch?v=1U5nOKh9OLQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=1U5nOKh9OLQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.5: RAPIDS für tabellarische Daten [[Video]](https://www.youtube.com/watch?v=KgoXuhG_kfs&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=KgoXuhG_kfs&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)


# Google CoLab-Anweisungen

Der folgende Code stellt sicher, dass Google CoLab ausgeführt wird und ordnet bei Bedarf Google Drive zu. Außerdem initialisieren wir das PyTorch-Gerät entweder auf GPU/MPS (falls verfügbar) oder CPU.


In [1]:
import copy
import torch

try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Nutzen Sie eine GPU oder MPS (Apple), sofern verfügbar. (siehe Modul 3.2)
import torch

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")


# Frühzeitiges Stoppen (siehe Modul 3.4)
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_model = copy.deepcopy(model.state_dict())
            self.best_loss = val_loss
            self.counter = 0
            self.status = f"Improvement found, counter reset to {self.counter}"
        else:
            self.counter += 1
            self.status = f"No improvement in the last {self.counter} epochs"
            if self.counter >= self.patience:
                self.status = f"Early stopping triggered after {self.counter} epochs."
                if self.restore_best_weights:
                    model.load_state_dict(self.best_model)
                return True
        return False

Note: not using Google CoLab
Using device: mps


# Teil 4.2: Trainingspläne für PyTorch

Lernratenpläne sind Mechanismen, die während des Trainings neuronaler Netzwerke verwendet werden, um die Lernrate im Laufe der Zeit anzupassen. Sie sind so konzipiert, dass die Lernrate im Verlauf des Trainings abnimmt, sodass das Netzwerk in den Anfangsphasen des Trainings, wenn die Gewichte wahrscheinlich weit von ihren optimalen Werten entfernt sind, große Anpassungen vornehmen und dann im Verlauf des Trainings kleinere Anpassungen vornehmen kann, um die Gewichte zu optimieren. Diese Anpassung trägt dazu bei, das Risiko zu verringern, den Mindestpunkt der Verlustfunktion zu überschreiten, und trägt dazu bei, die Konvergenz reibungsloser zu erreichen.

In PyTorch ist eines der Lernratenplanungstools die Klasse StepLR, die im Modul **torch.optim.lr_scheduler** zu finden ist. **StepLR** ist eine Art Lernratenplanung, die die Lernrate alle paar Epochen um einen bestimmten Faktor verringert. Dadurch kann die Lernrate schrittweise und nicht kontinuierlich verringert werden, was in einigen Fällen von Vorteil sein kann, da das Modell Zeit hat, sich in Bereichen der Verlustlandschaft „einzupendeln“, bevor die Lernrate weiter verringert wird.

StepLR verwendet drei Parameter:

* **Optimierer:** Der Optimierer, den Sie zum Trainieren Ihres Modells verwenden (z. B. SGD, Adam).
* **Schrittgröße:** Dies ist die Anzahl der Epochen, nach denen Sie die Lernrate reduzieren möchten. Wenn beispielsweise Schrittgröße = 10 ist, wird die Lernrate alle 10 Epochen reduziert.
* **gamma:** Dies ist der Faktor, um den die Lernrate bei jedem Schritt reduziert wird. Wenn beispielsweise gamma=0,1 ist, wird die Lernrate bei jedem Schritt mit 0,1 multipliziert, was sie effektiv um 90 % reduziert.

Der **StepLR**-Scheduler wird während der Trainingsschleife verwendet. Nach jedem Schritt des Optimierers (nach **optimizer.step()**) rufen Sie scheduler.step() auf, um die Lernrate entsprechend dem Zeitplan anzupassen.

Es ist erwähnenswert, dass die Wahl von **Schrittweite** und Gamma wichtig sein kann und möglicherweise basierend auf Ihrem spezifischen Problem und Datensatz angepasst werden muss. Eine zu große **Schrittweite** und die Lernrate kann sich möglicherweise nicht schnell genug verringern; eine zu kleine und sie kann sich zu schnell verringern. Ebenso kann ein Gamma, das zu nahe bei 1 liegt, die Lernrate möglicherweise nicht signifikant genug verringern, während ein zu kleines Gamma sie möglicherweise zu schnell verringert.

Wir wenden jetzt eine Lernrate auf das Beispiel der k-fachen Kreuzvalidierung aus dem vorherigen Abschnitt an.


In [2]:
import pandas as pd
from scipy.stats import zscore
from sklearn.model_selection import train_test_split

# Lesen des Datensatzes
df = read_csv(
    "https://data.heatonresearch.com/data/t81-558/jh-simple-dataset.csv",
    na_values=["NA", "?"],
)

# Dummies für den Job generieren
df = concat([df, get_dummies(df["job"], prefix="job", dtype=int)], axis=1)
df.drop("job", axis=1, inplace=True)

# Dummies für den Bereich generieren
df = concat([df, get_dummies(df["area"], prefix="area", dtype=int)], axis=1)
df.drop("area", axis=1, inplace=True)

# Dummies für Produkte generieren
df = concat([df, get_dummies(df["product"], prefix="product", dtype=int)], axis=1)
df.drop("product", axis=1, inplace=True)

# Fehlende Werte für Einkommen
med = df["income"].median()
df["income"] = df["income"].fillna(med)

# Bereiche standardisieren
df["income"] = zscore(df["income"])
df["aspect"] = zscore(df["aspect"])
df["save_rate"] = zscore(df["save_rate"])
df["subscriptions"] = zscore(df["subscriptions"])

Nachdem der Merkmalsvektor erstellt wurde, kann eine 5-fache Kreuzvalidierung durchgeführt werden, um Out-of-Sample-Vorhersagen zu generieren. Wir gehen von 500 Epochen aus und verwenden kein frühzeitiges Stoppen. Später werden wir sehen, wie wir eine optimalere Epochenanzahl schätzen können.


In [3]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# In PyTorch-Tensoren konvertieren
x_columns = df.columns.drop(["age", "id"])
x = torch.tensor(df[x_columns].values, dtype=torch.float32, device=device)
y = torch.tensor(df["age"].values, dtype=torch.float32, device=device).view(-1, 1)

# Legen Sie einen Zufallsstartwert für die Reproduzierbarkeit fest
torch.manual_seed(42)

# Kreuzvalidierung
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Parameter für vorzeitiges Stoppen
patience = 10

fold = 0
for train_idx, test_idx in kf.split(x):
    fold += 1
    print(f"Fold # {falten}")

    x_train, x_test = x[train_idx], x[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # PyTorch DataLoader
    train_dataset = TensorDataset(x_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Erstellen des Modells und des Optimierers
    model = nn.Sequential(
        nn.Linear(x.shape[1], 20),
        nn.ReLU(),
        nn.Linear(20, 10),
        nn.ReLU(),
        nn.Linear(10, 1),
    )
    model = torch.compile(model, backend="aot_eager").to(device)

    optimizer = optim.Adam(model.parameters())
    # Passen Sie die Lernrate alle 50 Epochen an
    scheduler = StepLR(optimizer, step_size=50, gamma=0.90)
    loss_fn = nn.MSELoss()

    # Variablen für frühzeitiges Stoppen
    best_loss = float("inf")
    early_stopping_counter = 0

    # Trainingsschleife
    EPOCHS = 500
    epoch = 0
    done = False
    es = EarlyStopping()

    while not done and epoch < EPOCHS:
        epoch += 1
        model.train()
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(x_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            optimizer.step()

        scheduler.step()  # Lernratenplan anwenden
        # Validierung
        model.eval()
        with torch.no_grad():
            val_output = model(x_test)
            val_loss = loss_fn(val_output, y_test)

        if es(model, val_loss):
            done = True

    print(
        f"Epoch {epoch}/{EPOCHS}, Validation Loss: " f"{val_loss.item()}, {es.status}"
    )

# Abschließende Bewertung
model.eval()
with torch.no_grad():
    oos_pred = model(x_test)
score = torch.sqrt(loss_fn(oos_pred, y_test)).item()
print(f"Faltungsbewertung (RMSE): {score}")

Fold #1


/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/autograd/__init__.py:394: UserWarning: Error detected in LinearBackward0. Traceback of forward call that caused the error:
  File "/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/nn/modules/container.py", line 215, in forward
    input = module(input)
 (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1702400227158/work/torch/csrc/autograd/python_anomaly_mode.cpp:119.)
  result = Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[2024-01-07 07:04:11,083] [0/2] torch._dynamo.exc: [WARNING] Backend compiler failed with a fake tensor exception at 
[2024-01-07 07:04:11,083] [0/2] torch._dynamo.exc: [WARNING]   File "/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/nn/modules/container.py", line 216, in forward
[2024-01-07 07:04:11,083] [0/2] torch._dynamo.exc: [WARNING]     return input
[2024-01-07 07:04:11,083] [

Epoch 199/500, Validation Loss: 0.5704283714294434, Early stopping triggered after 5 epochs.
Fold #2
Epoch 146/500, Validation Loss: 0.45389053225517273, Early stopping triggered after 5 epochs.
Fold #3
Epoch 165/500, Validation Loss: 0.7377966046333313, Early stopping triggered after 5 epochs.
Fold #4
Epoch 165/500, Validation Loss: 0.45836910605430603, Early stopping triggered after 5 epochs.
Fold #5
Epoch 137/500, Validation Loss: 1.2551432847976685, Early stopping triggered after 5 epochs.
Fold score (RMSE): 1.1172937154769897
